# Domain Adaptation

## Loading the Datasets

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define transformation pipeline for MNIST
transform_mnist = transforms.Compose([
    # TODO: resize to 32x32,                # match SVHN size
    # TODO: use 3 channels for Grayscale,   # convert to 3 channels (because SVHN images are not grayscale)
    # TODO: transform to tensor,
    transforms.Normalize((0.5,), (0.5,))
])

# Define transformation pipeline for SVHN
transform_svhn = transforms.Compose([
    # TODO: transform to tensor,
    transforms.Normalize((0.5,), (0.5,))
])

# Get train and test sets for MNIST
mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform_mnist)
mnist_test  = datasets.MNIST(root="./data", train=False, transform=transform_mnist)

# Get train and test sets for SVHN
svhn_train = datasets.SVHN(root="./data", split="train", download=True, transform=transform_svhn)
svhn_test  = datasets.SVHN(root="./data", split="test", download=True, transform=transform_svhn)

# Define data loaders
mnist_loader = None # TODO
mnist_test_loader = None # TODO

svhn_loader = None # TODO
svhn_test_loader = None # TODO

## Model Definition

In [ ]:
# Define a simple CNN for feature extraction
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            # TODO
        )

    def forward(self, x):
        return None # TODO

# Define a final MLP for classification
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            # TODO
        )

    def forward(self, x):
        return None # TODO

# Instantiate models and move them to right device
feature_extractor = None  # TODO
classifier = None         # TODO


Training on MNIST...
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done

Before adaptation:
MNIST accuracy: 0.9912
SVHN accuracy: 0.3368

Domain adaptation...
DA Epoch 1 done | Loss: 0.0205
DA Epoch 2 done | Loss: 0.0163
DA Epoch 3 done | Loss: 0.0145
DA Epoch 4 done | Loss: 0.0095
DA Epoch 5 done | Loss: 0.0094

After adaptation:
MNIST accuracy: 0.9932
SVHN accuracy: 0.3460


0.34603564843269824

## Training Loop

In [ ]:
# Define classification loss function and optimizer
criterion = None  # TODO
optimizer = optim.Adam(list(feature_extractor.parameters()) + list(classifier.parameters()), lr=1e-3)

# Define standard training loop to train the model over mnist
def train_mnist(epochs=5):
    # TODO: set models to train mode

    for epoch in range(epochs):
        for x, y in mnist_loader:
            # TODO

        print(f"Epoch {epoch+1} done")

# Define evaluation function
def evaluate(loader, name="dataset"):
    feature_extractor.eval()
    classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = classifier(feature_extractor(x))
            preds = preds.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    print(f"{name} accuracy: {acc:.4f}")
    return acc

## Domain Adaptation Training

The **Maximum Mean Discrepancy (MMD)** is a metric used to measure distance between two probability distributions based on samples.
We define a kernel function that acts as similarity measure:

$$K(x, y) = \exp\left(-\frac{\|x - y\|^2}{2\sigma^2}\right)$$

This kernel maps data into a high dimensional space to find patterns.
If two points are identical, the result is 1. As points get further apart, the result decays toward 0.
$\sigma$ controls the reach of the kernel: small sigma means points must be very close to be considered similar; a large sigma is more forgiving.

This metric is then used to compute the MMD Loss according to the following formula:

$$\text{MMD}^2(P, Q) = E[K(X, X')] + E[K(Y, Y')] - 2E[K(X, Y)]$$

The first term measures how similar is the first dataset to itself.
The second term measures how similar is the second dataset to itself.
The last term measures how similar is the first dataset to the second one.

**Intuition:** if x and y comes from the same distribution, the similarity between them should be roughly the same as the internal similarities, pushing the loss toward 0.

In [ ]:
# Define MMD Loss function

# Implements Radial Basis Function (RBF) or Gaussian Kernel
def gaussian_kernel(x, y, sigma=1.0):
    x = x.unsqueeze(1)
    y = y.unsqueeze(0)
    return None   # TODO

def mmd_loss(x, y):
    Kxx = None  # TODO: compute similarities between x and x
    Kyy = None  # TODO: compute similarities between y and y
    Kxy = None  # TODO: compute similarities between x and y
    return None # TODO


# Define training for Domain Adaptation
def train_domain_adaptation(epochs=5, lambda_mmd=0.5):
    # TODO: set models to train mode

    svhn_iter = iter(svhn_loader)

    for epoch in range(epochs):
        for mnist_x, mnist_y in mnist_loader:

            try:
                svhn_x, _ = next(svhn_iter)
            except StopIteration:
                svhn_iter = iter(svhn_loader)
                svhn_x, _ = next(svhn_iter)

            mnist_x, mnist_y = mnist_x.to(device), mnist_y.to(device)
            svhn_x = svhn_x.to(device)

            # TODO: clear out old gradients

            # source (MNIST)
            f_src = None  # TODO: extract features from mnist_x
            preds = None  # TODO: apply classifier
            cls_loss = None # TODO

            # target (SVHN)
            f_tgt = None  # TODO

            # MMD
            mmd = None  # TODO

            loss = None # TODO

            # TODO (backward pass and step)

        print(f"DA Epoch {epoch+1} done | Loss: {loss.item():.4f}")

## Final Pipeline

In [ ]:
# Run entire pipeline
print("\nTraining on MNIST...")
# TODO

print("\nBefore adaptation:")
# TODO: evaluate over MNIST
# TODO: evaluate over SVHN

print("\nDomain adaptation...")
# TODO

print("\nAfter adaptation:")
# TODO